# Interpreting pretrained MOMENT with WinTSR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/pretrained_moment.ipynb)

Use [MOMENT](https://arxiv.org/abs/2402.03885), an open time-series foundation model, to make a real zero-shot forecast and reveal which observations drove it with WinTSR. We use the smallest [MOMENT-1 checkpoint on Hugging Face](https://huggingface.co/AutonLab/MOMENT-1-small), at about 40M parameters.

In [ ]:
%pip install -q tslens momentfm

## 1. Load MOMENT and make a zero-shot forecast

MOMENT's position embeddings are fixed to a sequence length of exactly 512. This is a hard constraint: every input must be padded or truncated to 512 before inference. Our ETTh2 window has 512 real oil-temperature observations, so no padding mask is needed; when omitted, MOMENT uses an all-ones mask internally.

In [ ]:
import pandas as pd
import torch
from momentfm import MOMENTPipeline

DATA_URL = "https://raw.githubusercontent.com/WenWeiTHU/TimeSeriesDatasets/refs/heads/main/ETT-small/ETTh2.csv"
CONTEXT_LEN, FORECAST_HORIZON = 512, 96
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

series = torch.tensor(pd.read_csv(DATA_URL)["OT"].dropna().to_numpy(), dtype=torch.float32)
starts = (0, 24)
inputs = torch.stack([series[s : s + CONTEXT_LEN] for s in starts]).unsqueeze(-1).to(device)
ground_truth = series[CONTEXT_LEN : CONTEXT_LEN + FORECAST_HORIZON]

model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-small",
    model_kwargs={"task_name": "forecasting", "forecast_horizon": 96},
)
model.init()
model.to(device).eval()

x_enc = inputs[0].permute(1, 0).unsqueeze(0)
with torch.inference_mode():
    output = model(x_enc=x_enc)
prediction = output.forecast[0, 0].detach().cpu()
print(f"device: {device} | x_enc: {tuple(x_enc.shape)} | forecast: {tuple(output.forecast.shape)}")

In [ ]:
import matplotlib.pyplot as plt

context = inputs[0, :, 0].detach().cpu()
forecast_steps = torch.arange(CONTEXT_LEN, CONTEXT_LEN + FORECAST_HORIZON)
plt.figure(figsize=(12, 4))
plt.plot(torch.arange(CONTEXT_LEN), context, label="lookback", linewidth=1)
plt.plot(forecast_steps, ground_truth, label="ground truth", linewidth=2)
plt.plot(forecast_steps, prediction, label="MOMENT forecast", linewidth=2)
plt.axvline(CONTEXT_LEN - 1, color="0.4", linestyle="--", linewidth=1)
plt.xlabel("hour")
plt.ylabel("oil temperature (OT)")
plt.title("MOMENT zero-shot forecast on ETTh2")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Adapt channel-first MOMENT to WinTSR

MOMENT consumes `(batch, n_channels, seq_len)`, while WinTSR attributes `(batch, seq_len, n_features)`. This adapter only swaps those axes and extracts the forecast tensor.

In [ ]:
class MomentWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        x_enc = x.permute(0, 2, 1)
        out = self.model(x_enc=x_enc).forecast
        return out.permute(0, 2, 1)

wrapped = MomentWrapper(model).eval()
with torch.inference_mode():
    print("adapter output:", tuple(wrapped(inputs).shape))

## 3. Attribute the forecast

We explain two nearby ETTh2 windows. `threshold=0.5` keeps every time step in stage one, then skips the lower half during WinTSR's feature-level stage.

In [ ]:
from tslens import WinTSR

baselines = torch.zeros_like(inputs)
attr = WinTSR(wrapped).attribute(
    inputs=inputs,
    baselines=baselines,
    threshold=0.5,
    show_progress=True,
)
print("attributions:", tuple(attr.shape), "= (batch, horizon, time, feature)")

## 4. See which time steps mattered

As in the Timer tutorial, the input and attribution share an x-axis. We average absolute attribution over all 96 forecast horizons for sample 0.

In [ ]:
recent = inputs[0, :, 0].detach().cpu()
saliency = attr[0].abs().mean(dim=0).squeeze(-1).detach().cpu()
steps = torch.arange(-CONTEXT_LEN, 0)

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True, height_ratios=(2, 1))
axes[0].plot(steps, recent, color="tab:blue", linewidth=1)
axes[0].set_ylabel("OT")
axes[0].set_title("MOMENT input and WinTSR attribution")
axes[0].grid(alpha=0.25)
axes[1].fill_between(steps, saliency, color="tab:orange", alpha=0.8)
axes[1].plot(steps, saliency, color="tab:orange", linewidth=0.8)
axes[1].set_xlabel("hours before forecast")
axes[1].set_ylabel("attribution")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Read the pattern, not a story

Look for concentration near the forecast boundary and peaks at repeating lags such as 24 hours. Treat those as patterns to test, not conclusions guaranteed in advance: attribution measures sensitivity to the chosen zero replacement, which may be outside the normal oil-temperature range.

## Next steps

- Try a seasonal or local-mean baseline and check whether the important lags persist.
- Index `attr[:, h]` instead of averaging to explain one forecast horizon.
- Build `inputs` from `HUFL`, `HULL`, and `OT` to obtain a genuine `(seq_len, n_features)` multivariate saliency heatmap; expect the extra feature-level perturbations to take longer.

If this was useful, please star the [repository](https://github.com/khairulislam/tslens). Please cite the following if you use our work:

```bibtex
@article{goswami2024moment,
  title={MOMENT: A Family of Open Time-series Foundation Models},
  author={Goswami, Mononito and Szafer, Konrad and Choudhry, Arjun and Cai, Yifu and Li, Shuo and Dubrawski, Artur},
  journal={arXiv preprint arXiv:2402.03885},
  year={2024}
}

@article{islam2024wintsr,
  title={WinTSR: A Windowed Temporal Saliency Rescaling Method for Interpreting Time Series Deep Learning Models},
  author={Islam, Md Khairul and Fox, Judy},
  journal={arXiv preprint arXiv:2412.04532},
  year={2024}
}
```